# 01 - Cek dataset

Pastikan struktur folder, nama kelas, dan bounding box sudah benar.

In [ ]:
%load_ext autoreload
%autoreload 2

import glob, os, random, yaml
import cv2
import matplotlib.pyplot as plt

In [ ]:
meta = yaml.safe_load(open("dataset/person/data.yaml", encoding="utf-8"))
print(meta)

names = meta["names"]
if isinstance(names, list):
    names = {i: n for i, n in enumerate(names)}

imgs = glob.glob("dataset/person/images/**/*.jpg", recursive=True)
imgs += glob.glob("dataset/person/images/**/*.png", recursive=True)
print("total gambar:", len(imgs))
assert imgs, "Tidak ada gambar - cek struktur dataset/"

In [ ]:
def load_boxes(lbl_path, w, h):
    boxes = []
    if not os.path.exists(lbl_path):
        return boxes
    for line in open(lbl_path):
        p = line.split()
        if len(p) < 5:
            continue
        c = int(float(p[0]))
        x, y, bw, bh = map(float, p[1:5])
        boxes.append((c, (x - bw/2)*w, (y - bh/2)*h, (x + bw/2)*w, (y + bh/2)*h))
    return boxes

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, path in zip(axes, random.sample(imgs, 3)):
    img = cv2.imread(path)
    h, w = img.shape[:2]
    lbl = path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"
    for c, x1, y1, x2, y2 in load_boxes(lbl, w, h):
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(img, names.get(c, str(c)), (int(x1), int(y1) - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.show()